# PRÉ-PROJETO DE PESQUISA OPERACIONAL — Pyomo com HiGHS, CBC ou Gurobi

## Otimização Operacional do Agendamento de Bombas e Distribuição de Água Tratada via Programação Linear Inteira Mista (MILP)

- Escolha a resolução temporal na **Seção 2** alterando `MODO_TEMPORAL` (`"hora"` para 24 períodos ou `"minuto"` para 1.440 períodos).
- Escolha o solver na **Seção 3** alterando `SOLVER` (`"highs"`, `"highs_base"`, `"cbc"` ou `"gurobi"`).
- **Melhorias aplicadas:** formulação enxuta (`q` como `Expression`, limites de `V`/`f_pr`/`f_rd` nos `bounds`, arcos pré-indexados + `quicksum`), warm-start hierárquico opcional no modo minuto (`USAR_WARM_START`), acompanhamento temporal da otimização para **HiGHS e CBC** (tabela + gráficos de incumbente/limite inferior e gap), e proteção para execução independente do cenário-base.

## 1. Instalação de Dependências

Instala ou atualiza o Pyomo e os solvers compatíveis (`highspy`, `cbcbox` e `gurobipy`). Execute apenas na primeira configuração do ambiente.

In [ ]:
%pip install --upgrade "pyomo>=6.8" "highspy>=1.8" cbcbox gurobipy

## 2. Configurações Gerais, Topologia e Dados do Sistema

Centraliza todas as chaves de controle (`MODO_TEMPORAL`, cenário-base, acompanhamento de progresso e warm-start), a topologia da rede (arcos bomba → reservatório e reservatório → setor), tarifas horossazonais, parâmetros físicos e curva de demanda diária.

In [ ]:
from collections import defaultdict
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pyomo.environ import (
    ConcreteModel, Set, Var, Expression, Objective, Constraint, Binary,
    NonNegativeReals, RangeSet, minimize, value, SolverFactory,
    TerminationCondition, quicksum
)
import pyomo.environ as pyo

# ==============================================================================
# CONFIGURAÇÕES PRINCIPAIS DE EXECUÇÃO (altere aqui)
# ==============================================================================
# Resolução temporal do modelo: "hora" (24 intervalos) ou "minuto" (1.440 intervalos)
MODO_TEMPORAL = "minuto"

# Cenário-base (tarifa fixa): defina False para pular o cálculo e os gráficos do cenário-base.
GERAR_CENARIO_BASE = True
TEMPO_MAX_BASE_S = 300  # Limite próprio (s) para o cenário-base

# Acompanhamento da evolução do solver ao longo do tempo (HiGHS via callback; CBC via log)
ACOMPANHAR_PROGRESSO = True
INTERVALO_ACOMPANHAMENTO_MIN = 5  # Ex.: 5 ou 1 (no modo "hora" aceita valores menores, ex.: 0.1 = 6 s)

# Warm-start hierárquico (ativo automaticamente apenas no modo "minuto"):
# resolve primeiro o modelo agregado (ex.: 60 min = 24 passos) e fornece a solução inicial viável.
USAR_WARM_START = True
PASSO_WARM_START_MIN = 60    # Deve dividir 60 (ex.: 60 ou 15)
TEMPO_MAX_WARM_START_S = 60  # 60 s permite ao modelo horário convergir ao ótimo (~R$ 10.724–10.733)

# Penalidade por partida (R$/acionamento). Mantenha 0.0 para máximo desempenho.
PENALIDADE_PARTIDA = 0.0

# Tempo mínimo contínuo que uma bomba deve permanecer ligada (em minutos).
# 0 mantém exatamente a formulação matemática original.
TEMPO_MIN_LIGADO_MIN = 0
# ==============================================================================

if MODO_TEMPORAL not in ("hora", "minuto"):
    raise ValueError(
        f"MODO_TEMPORAL inválido: {MODO_TEMPORAL!r}. Escolha 'hora' ou 'minuto'."
    )

# 1. Conjuntos de Índices derivados automaticamente do modo selecionado
T = 24
horas = range(T)
MINUTOS_POR_DIA = T * 60
DURACAO_PASSO_MIN = 60 if MODO_TEMPORAL == "hora" else 1
N_PASSOS = MINUTOS_POR_DIA // DURACAO_PASSO_MIN
FATOR_HORA = DURACAO_PASSO_MIN / 60.0
passos = range(N_PASSOS)
DESCRICAO_MODO = (
    f"passos de 1 hora ({N_PASSOS} períodos)"
    if MODO_TEMPORAL == "hora"
    else f"passos de 1 minuto ({N_PASSOS} períodos)"
)
bombas = range(1, 7)
reservatorios = range(1, 6)
setores = range(1, 13)

# 2. Topologia do Sistema (Arcos permitidos conforme o diagrama)
arcos_PR = [
    (1, 1),
    (2, 1), (2, 2), (2, 3),
    (3, 2),
    (4, 3), (4, 4),
    (5, 5),
    (6, 4), (6, 5)
]

arcos_RD = [
    (1, 1), (1, 2), (1, 3),
    (2, 2), (2, 4), (2, 5),
    (3, 4), (3, 6), (3, 7),
    (4, 4), (4, 8), (4, 9), (4, 10), (4, 12),
    (5, 7), (5, 10), (5, 11), (5, 12)
]

# Pré-indexação dos arcos para evitar buscas lineares durante a montagem do modelo
arcos_saida_bomba = defaultdict(list)
arcos_entrada_res = defaultdict(list)
for p, r in arcos_PR:
    arcos_saida_bomba[p].append((p, r))
    arcos_entrada_res[r].append((p, r))

arcos_saida_res = defaultdict(list)
arcos_entrada_setor = defaultdict(list)
for r, d in arcos_RD:
    arcos_saida_res[r].append((r, d))
    arcos_entrada_setor[d].append((r, d))

# 3. Parâmetros Econômicos (Tarifa Horossazonal - R$/kWh)
# Tarifa fora de ponta: R$ 0.4525 | Tarifa de ponta (18h00 às 20h59): R$ 1.4048
C = {t: 1.4048 if 18 <= t < 21 else 0.4525 for t in horas}

# 4. Parâmetros das bombas: (potência em kW, vazão mínima e máxima em m³/h)
param_bombas = {
    1: (240, 1170, 2340),
    2: (450, 2180, 4370),
    3: (240, 1170, 2340),
    4: (290, 1400, 2810),
    5: (190,  940, 1870),
    6: (240, 1170, 2340)
}
Q_ETA_max = 9400  # m³/h; limite de vazão agregada da ETA

# 5. Parâmetros dos Reservatórios: (volume mínimo, máximo e inicial), em m³
param_res = {
    1: (2100, 14000, 4200),
    2: (1400,  9400, 2800),
    3: (1400,  9400, 2800),
    4: (1400,  9400, 2800),
    5: ( 700,  4700, 1400)
}

# 6. Demandas-base dos setores (m³/h); total: 5195 m³/h
base_demanda = {
    1: 312,  2: 468,  3: 390,  4: 624,
    5: 390,  6: 312,  7: 546,  8: 468,
    9: 312, 10: 702, 11: 390, 12: 281
}

# Curva horária de demanda de verão; cada fator multiplica a demanda-base dos setores.
curva_verao = [
    0.8, 0.5, 0.3, 0.4, 0.6, 1.0,   # 00h às 05h: menor consumo na madrugada
    1.8, 2.0, 1.9, 1.7, 1.5, 1.5,   # 06h às 11h: pico da manhã
    1.4, 1.5, 1.6, 1.7, 1.8, 2.9,   # 12h às 17h: consumo alto e crescente
    3.4, 3.0, 2.6, 2.0, 1.5, 1.0    # 18h às 23h: pico máximo às 18h
]

D = {
    (d, t): round(base_demanda[d] * curva_verao[t], 2)
    for d in setores for t in horas
}

# Indicadores de demanda e armazenamento calculados para o horizonte diário.
demanda_horaria_total = [sum(D[d, h] for d in setores) for h in horas]
volume_demanda_diario = sum(demanda_horaria_total)
demanda_media_horaria = volume_demanda_diario / T
hora_pico = max(horas, key=lambda h: demanda_horaria_total[h])
demanda_pico_horaria = demanda_horaria_total[hora_pico]
armazenamento_util_total = sum(
    param_res[r][1] - param_res[r][0] for r in reservatorios
)

saldo_acumulado = [0.0]
for demanda in demanda_horaria_total:
    saldo_acumulado.append(
        saldo_acumulado[-1] + demanda_media_horaria - demanda
    )
armazenamento_util_estimado = max(saldo_acumulado) - min(saldo_acumulado)

def clamp(valor, limite_inf, limite_sup):
    """Garante que um valor numérico fique estritamente dentro dos bounds [limite_inf, limite_sup]."""
    return min(limite_sup, max(limite_inf, float(valor)))

def contar_partidas(modelo_solucao):
    """Conta o número total de transições 0 -> 1 nas bombas (funciona para modo hora ou minuto)."""
    total = 0
    for p in bombas:
        anterior = 0
        for t in modelo_solucao.T:
            atual = 1 if value(modelo_solucao.x[p, t]) > 0.5 else 0
            if atual == 1 and anterior == 0:
                total += 1
            anterior = atual
    return total

print(f"Modo selecionado: {MODO_TEMPORAL.upper()} — {DESCRICAO_MODO}")
print(f"Soma das demandas-base: {sum(base_demanda.values())} m³/h")
print(f"Demanda média horária: {demanda_media_horaria:.2f} m³/h")
print(f"Pico horário: {demanda_pico_horaria:.2f} m³/h às {hora_pico:02d}h")
print(f"Volume de demanda diário: {volume_demanda_diario:.2f} m³/dia")
print(f"Volume útil agregado dos reservatórios: {armazenamento_util_total:.0f} m³")
print(
    "Armazenamento útil estimado pelo perfil: "
    f"{armazenamento_util_estimado:.2f} m³ "
    "(estimativa agregada, com bombeamento constante na demanda média)"
)

## 3. Configuração do Solver e Construção do Modelo MILP

Define o solver (`SOLVER`), seus parâmetros de execução, o construtor enxuto do modelo em Pyomo (`construir_modelo_eta`) e as rotinas de warm-start hierárquico (`gerar_warm_start`), construindo o modelo principal no modo escolhido.

In [ ]:
import time

# Altere somente esta variável: "highs", "highs_base", "cbc" ou "gurobi".
SOLVER = "highs"

SOLVERS = {
    "highs": {
        "factory": "appsi_highs",
        "options": {
            "time_limit": 3600,  # Em segundos (3600 = 1 hora)
            "mip_rel_gap": 0.003,
            "threads": 0,
            "random_seed": 0,
            "mip_detect_symmetry": True,
            "mip_heuristic_effort": 0.10,
            "mip_heuristic_run_shifting": True,
            "parallel": "choose",
        },
    },
    "highs_base": {
        "factory": "appsi_highs",
        "options": {
            "time_limit": 300,
            "mip_rel_gap": 0.003,
            "threads": 0,
            "random_seed": 0,
            "parallel": "choose",
        },
    },
    "cbc": {
        "factory": "cbc",
        "options": {
            "seconds": 3600,
            "ratio": 0.003,
            "threads": 0,
            "randomSeed": 0,
        },
    },
    "gurobi": {
        "factory": "gurobi_direct",
        "options": {
            "TimeLimit": 300,
            "MIPGap": 0.003,
            "Threads": 0,
            "Seed": 0,
        },
    },
}

if SOLVER not in SOLVERS:
    raise ValueError(
        f"SOLVER inválido: {SOLVER!r}. Escolha entre {', '.join(SOLVERS)}."
    )

def obter_gap_alvo_pct():
    """Retorna o gap-alvo configurado para o solver selecionado, em porcentagem."""
    opts = SOLVERS[SOLVER]["options"]
    for chave in ("mip_rel_gap", "ratio", "MIPGap"):
        if chave in opts and opts[chave] is not None:
            return float(opts[chave]) * 100.0
    return None

def criar_solver(verbose=False, time_limit_override=None):
    config = SOLVERS[SOLVER]
    if SOLVER == "cbc":
        import cbcbox
        solver = SolverFactory(
            config["factory"], executable=cbcbox.cbc_bin_path()
        )
    else:
        solver = SolverFactory(config["factory"])
    if not solver.available(exception_flag=False):
        raise RuntimeError(
            f"O solver {SOLVER!r} não está disponível. Instale/configure o solver "
            "selecionado e, para Gurobi, verifique também a licença."
        )

    for option, setting in config["options"].items():
        solver.options[option] = setting

    if time_limit_override is not None:
        if SOLVER in ("highs", "highs_base"):
            solver.options["time_limit"] = time_limit_override
        elif SOLVER == "cbc":
            solver.options["seconds"] = time_limit_override
        else:
            solver.options["TimeLimit"] = time_limit_override

    if SOLVER in ("highs", "highs_base"):
        solver.options["output_flag"] = verbose
    elif SOLVER == "cbc":
        solver.options["logLevel"] = 1 if verbose else 0
    else:
        solver.options["OutputFlag"] = int(verbose)
    return solver

def construir_modelo_eta(
    passo_min=1,
    penalidade_partida=0.0,
    tempo_min_ligado_min=0,
    tarifa_fixa=False,
    vazao_ponta_min_m3h=None,
    nome="Agendamento_Bombas_MILP",
):
    """Constrói o MILP enxuto e pré-indexado para qualquer passo divisor de 60 min (ex.: 60 ou 1)."""
    if 60 % passo_min != 0:
        raise ValueError(f"passo_min ({passo_min}) deve ser um divisor de 60.")

    n_passos = MINUTOS_POR_DIA // passo_min
    fator_h = passo_min / 60.0

    m = ConcreteModel(name=nome)
    m.P = RangeSet(1, 6)
    m.R = RangeSet(1, 5)
    m.D = RangeSet(1, 12)
    m.T = RangeSet(0, n_passos - 1)
    m.ARCO_PR = Set(dimen=2, initialize=arcos_PR)
    m.ARCO_RD = Set(dimen=2, initialize=arcos_RD)

    # f_pr e f_rd estão sempre em m³/min; V em m³; x indica acionamento da bomba.
    m.x = Var(m.P, m.T, domain=Binary)
    m.f_pr = Var(
        m.ARCO_PR, m.T,
        domain=NonNegativeReals,
        bounds=lambda mod, p, r, t: (0.0, param_bombas[p][2] / 60.0),
    )
    m.f_rd = Var(
        m.ARCO_RD, m.T,
        domain=NonNegativeReals,
        bounds=lambda mod, r, d, t: (0.0, D[d, (t * passo_min) // 60] / 60.0),
    )

    def limites_volume(mod, r, t):
        v_min, v_max, v_ini = param_res[r]
        lb = max(v_min, v_ini) if t == n_passos - 1 else v_min
        return (lb, v_max)

    m.V = Var(m.R, m.T, domain=NonNegativeReals, bounds=limites_volume)

    # Expressão simbólica para vazão da bomba em m³/min (elimina variáveis q e igualdades fluxo_bomba)
    m.q = Expression(
        m.P, m.T,
        rule=lambda mod, p, t: quicksum(mod.f_pr[p, r, t] for _, r in arcos_saida_bomba[p]),
    )

    # Detecção de partidas (ativada apenas se penalidade_partida > 0 ou tempo_min_ligado_min > passo_min)
    passos_min_ligado = int(np.ceil(tempo_min_ligado_min / passo_min)) if tempo_min_ligado_min > 1 else 0
    usar_partidas = (penalidade_partida > 0) or (passos_min_ligado > 1)
    if usar_partidas:
        m.y = Var(m.P, m.T, domain=NonNegativeReals, bounds=(0.0, 1.0))

        def regra_partida(mod, p, t):
            if t == 0:
                return mod.y[p, 0] >= mod.x[p, 0]
            return mod.y[p, t] >= mod.x[p, t] - mod.x[p, t - 1]

        m.deteccao_partida = Constraint(m.P, m.T, rule=regra_partida)

    if passos_min_ligado > 1:
        def regra_tempo_min_ligado(mod, p, t):
            inicio = max(0, t - passos_min_ligado + 1)
            return quicksum(mod.y[p, tau] for tau in range(inicio, t + 1)) <= mod.x[p, t]

        m.tempo_min_ligado = Constraint(m.P, m.T, rule=regra_tempo_min_ligado)

    # Objetivo: minimizar custo diário de energia (ou kWh no cenário-base)
    if tarifa_fixa:
        custo_base_expr = quicksum(
            param_bombas[p][0] * fator_h * m.x[p, t]
            for p in m.P for t in m.T
        )
    else:
        custo_base_expr = quicksum(
            C[(t * passo_min) // 60] * param_bombas[p][0] * fator_h * m.x[p, t]
            for p in m.P for t in m.T
        )

    if penalidade_partida > 0:
        m.obj = Objective(
            expr=custo_base_expr
            + penalidade_partida * quicksum(m.y[p, t] for p in m.P for t in m.T),
            sense=minimize,
        )
    else:
        m.obj = Objective(expr=custo_base_expr, sense=minimize)

    # Limites operacionais das bombas
    m.limite_min_bomba = Constraint(
        m.P, m.T,
        rule=lambda mod, p, t: mod.q[p, t] >= (param_bombas[p][1] / 60.0) * mod.x[p, t],
    )
    m.limite_max_bomba = Constraint(
        m.P, m.T,
        rule=lambda mod, p, t: mod.q[p, t] <= (param_bombas[p][2] / 60.0) * mod.x[p, t],
    )

    # Capacidade máxima da ETA
    m.capacidade_ETA = Constraint(
        m.T,
        rule=lambda mod, t: quicksum(mod.q[p, t] for p in mod.P) <= Q_ETA_max / 60.0,
    )

    # Atendimento da demanda
    m.demanda = Constraint(
        m.D, m.T,
        rule=lambda mod, d, t: quicksum(
            mod.f_rd[r, d, t] for r, _ in arcos_entrada_setor[d]
        ) == D[d, (t * passo_min) // 60] / 60.0,
    )

    # Balanço hídrico dos reservatórios
    m.balanco = Constraint(
        m.R, m.T,
        rule=lambda mod, r, t: mod.V[r, t] ==
            (param_res[r][2] if t == 0 else mod.V[r, t - 1])
            + passo_min * quicksum(mod.f_pr[p, r, t] for p, _ in arcos_entrada_res[r])
            - passo_min * quicksum(mod.f_rd[r, d, t] for _, d in arcos_saida_res[r]),
    )

    if vazao_ponta_min_m3h is not None:
        passos_ponta = [t for t in range(n_passos) if (t * passo_min) // 60 in vazao_ponta_min_m3h]
        m.vazao_ponta = Constraint(
            passos_ponta,
            rule=lambda mod, t: quicksum(mod.q[p, t] for p in mod.P)
            >= vazao_ponta_min_m3h[(t * passo_min) // 60] / 60.0 - 1e-6,
        )

    return m

def aplicar_warm_start(modelo_destino, modelo_agregado, passo_destino_min, passo_agregado_min):
    """Expande a solução viável do modelo agregado para o modelo de passo mais fino."""
    fator_expansao = passo_agregado_min // passo_destino_min
    n_passos_dest = MINUTOS_POR_DIA // passo_destino_min

    for p in bombas:
        anterior = 0
        for t in range(n_passos_dest):
            t_ag = t // fator_expansao
            x_val = int(round(value(modelo_agregado.x[p, t_ag])))
            modelo_destino.x[p, t].value = x_val
            if hasattr(modelo_destino, "y"):
                modelo_destino.y[p, t].value = 1.0 if (x_val == 1 and anterior == 0) else 0.0
            anterior = x_val

    for p, r in arcos_PR:
        ub = param_bombas[p][2] / 60.0
        for t in range(n_passos_dest):
            modelo_destino.f_pr[p, r, t].value = clamp(
                value(modelo_agregado.f_pr[p, r, t // fator_expansao]), 0.0, ub
            )

    for r, d in arcos_RD:
        for t in range(n_passos_dest):
            ub = D[d, (t * passo_destino_min) // 60] / 60.0
            modelo_destino.f_rd[r, d, t].value = clamp(
                value(modelo_agregado.f_rd[r, d, t // fator_expansao]), 0.0, ub
            )

    for r in reservatorios:
        v_min, v_max, v_ini = param_res[r]
        v_atual = float(v_ini)
        for t in range(n_passos_dest):
            entrada = sum(modelo_destino.f_pr[p, r, t].value for p, _ in arcos_entrada_res[r])
            saida = sum(modelo_destino.f_rd[r, d, t].value for _, d in arcos_saida_res[r])
            v_atual = v_atual + passo_destino_min * (entrada - saida)
            lb = max(v_min, v_ini) if t == n_passos_dest - 1 else v_min
            modelo_destino.V[r, t].value = clamp(v_atual, lb, v_max)

def gerar_warm_start(modelo_destino, tarifa_fixa=False, vazao_ponta_min_m3h=None):
    """Resolve o modelo agregado (PASSO_WARM_START_MIN) e injeta a solução inicial quando aplicável."""
    if not USAR_WARM_START:
        return False
    if DURACAO_PASSO_MIN >= PASSO_WARM_START_MIN:
        print(
            f"[WARM-START] Modo {MODO_TEMPORAL} ativo (passo {DURACAO_PASSO_MIN} min); "
            "warm-start agregado não é necessário."
        )
        return False
    if PASSO_WARM_START_MIN % DURACAO_PASSO_MIN != 0:
        raise ValueError(
            f"PASSO_WARM_START_MIN ({PASSO_WARM_START_MIN}) deve ser múltiplo de DURACAO_PASSO_MIN ({DURACAO_PASSO_MIN})."
        )

    t_ws = time.perf_counter()
    modelo_ws = construir_modelo_eta(
        passo_min=PASSO_WARM_START_MIN,
        penalidade_partida=PENALIDADE_PARTIDA,
        tempo_min_ligado_min=TEMPO_MIN_LIGADO_MIN,
        tarifa_fixa=tarifa_fixa,
        vazao_ponta_min_m3h=vazao_ponta_min_m3h,
        nome=f"WarmStart_{PASSO_WARM_START_MIN}min",
    )
    solver_ws = criar_solver(verbose=False, time_limit_override=TEMPO_MAX_WARM_START_S)
    try:
        res_ws = solver_ws.solve(modelo_ws, load_solutions=True)
    except RuntimeError as erro_ws:
        print(f"[WARM-START] Nenhuma solução viável encontrada em {TEMPO_MAX_WARM_START_S} s ({erro_ws}).")
        return False

    obj_ws = value(modelo_ws.obj, exception=False)
    if obj_ws is None:
        print(f"[WARM-START] Sem solução viável (status: {res_ws.solver.termination_condition}).")
        return False

    aplicar_warm_start(modelo_destino, modelo_ws, DURACAO_PASSO_MIN, PASSO_WARM_START_MIN)
    duracao_ws = time.perf_counter() - t_ws
    unidade = "kWh" if tarifa_fixa else "R$"
    print(
        f"[WARM-START {PASSO_WARM_START_MIN} min -> {DURACAO_PASSO_MIN} min] "
        f"Solução inicial viável gerada em {duracao_ws:.2f} s "
        f"(status: {res_ws.solver.termination_condition} | objetivo: {obj_ws:.2f} {unidade})."
    )
    return True

# Construção do modelo principal no modo selecionado (hora ou minuto)
t_construcao_ini = time.perf_counter()
model = construir_modelo_eta(
    passo_min=DURACAO_PASSO_MIN,
    penalidade_partida=PENALIDADE_PARTIDA,
    tempo_min_ligado_min=TEMPO_MIN_LIGADO_MIN,
    tarifa_fixa=False,
    nome=f"Agendamento_Bombas_MILP_{MODO_TEMPORAL.upper()}",
)

print(
    f"Solver selecionado: {SOLVER.upper()} | "
    f"Modelo Pyomo ({DESCRICAO_MODO}) construído em {time.perf_counter() - t_construcao_ini:.2f} s."
)

## 4. Cenário-Base com Tarifa Fixa (Opcional)

Calcula a operação de referência sem sinal de preço horário (mantendo vazão mínima na ponta igual à demanda agregada, limitada à capacidade da ETA) e recalcula o custo com as tarifas reais. Esta célula pode ser pulada ou desativada (`GERAR_CENARIO_BASE = False`) sem afetar o restante do notebook.

In [ ]:
# O cenário-base é opcional; a otimização tarifária não depende dele.
modelo_base = None
resultado_base = None
status_base = None
consumo_base_kwh = None
custo_base_real = None
partidas_base = None
vazao_base_ponta = None
cenario_base_disponivel = False

if GERAR_CENARIO_BASE:
    horas_ponta = [h for h in horas if C[h] == max(C.values())]
    vazao_base_ponta = {
        h: min(sum(D[d, h] for d in setores), Q_ETA_max)
        for h in horas_ponta
    }

    modelo_base = construir_modelo_eta(
        passo_min=DURACAO_PASSO_MIN,
        penalidade_partida=PENALIDADE_PARTIDA,
        tempo_min_ligado_min=TEMPO_MIN_LIGADO_MIN,
        tarifa_fixa=True,
        vazao_ponta_min_m3h=vazao_base_ponta,
        nome=f"Cenario_Base_Tarifa_Fixa_{MODO_TEMPORAL.upper()}",
    )

    ws_base_ok = gerar_warm_start(
        modelo_base, tarifa_fixa=True, vazao_ponta_min_m3h=vazao_base_ponta
    )

    solver_base = criar_solver(verbose=False, time_limit_override=TEMPO_MAX_BASE_S)
    try:
        solve_kwargs_base = {"warmstart": True} if ws_base_ok else {}
        resultado_base = solver_base.solve(modelo_base, **solve_kwargs_base)
        status_base = resultado_base.solver.termination_condition
    except RuntimeError as erro_base:
        status_base = f"Erro do solver: {erro_base}"

    obj_base = value(modelo_base.obj, exception=False)
    if obj_base is None:
        print(
            "O cenário-base não encontrou solução viável. "
            f"Status: {status_base}. Comparação e gráficos do cenário-base serão ignorados."
        )
    else:
        cenario_base_disponivel = True
        consumo_base_kwh = sum(
            param_bombas[p][0] * FATOR_HORA * value(modelo_base.x[p, t])
            for p in modelo_base.P for t in modelo_base.T
        )
        custo_base_real = sum(
            C[(t * DURACAO_PASSO_MIN) // 60] * param_bombas[p][0] * FATOR_HORA * value(modelo_base.x[p, t])
            for p in modelo_base.P for t in modelo_base.T
        )
        partidas_base = contar_partidas(modelo_base)
        print(f"[CENÁRIO-BASE: TARIFA FIXA — {DESCRICAO_MODO}]")
        print("Status:", status_base)
        print("Vazão-alvo na ponta (m³/h):", vazao_base_ponta)
        print(f"Consumo de energia: {consumo_base_kwh:.2f} kWh")
        print(f"Custo dessa programação com as tarifas reais: R$ {custo_base_real:.2f}")
        print(f"Total de partidas das bombas: {partidas_base}")
else:
    print("Cálculo do cenário-base desativado; a otimização tarifária continuará normalmente.")

## 5. Otimização Tarifária, Acompanhamento e Comparação de Custos

Resolve o MILP principal com warm-start opcional, coleta a evolução temporal do solver (**HiGHS** via callback nativo e **CBC** via leitura do log de Branch-and-Bound), exibe a tabela de progresso e compara os resultados com o cenário-base quando disponível.

In [ ]:
# Otimização tarifária, acompanhamento temporal (HiGHS e CBC) e comparação de resultados
import os
import re
import tempfile

historico_progresso = []
tabela_progresso = None

# Warm-start opcional a partir do modelo agregado (ativo automaticamente apenas no modo "minuto")
warm_start_pronto = gerar_warm_start(model, tarifa_fixa=False)

solver = criar_solver(verbose=True)

tempo_inicio = time.perf_counter()
highs_model = None
acompanhar_com_highs = ACOMPANHAR_PROGRESSO and SOLVER in ("highs", "highs_base")
melhor_incumbente_info = {"tempo_s": None, "custo": None}

def numero_finito(valor):
    if valor is None:
        return None
    try:
        numero = float(valor)
    except (TypeError, ValueError):
        return None
    return numero if np.isfinite(numero) else None

def base_esta_disponivel():
    """Verifica com segurança se os dados do cenário-base existem (mesmo se a célula 4 não foi executada)."""
    return (
        bool(globals().get("cenario_base_disponivel", False))
        and globals().get("modelo_base") is not None
        and globals().get("consumo_base_kwh") is not None
        and globals().get("custo_base_real") is not None
    )

def extrair_progresso_log_cbc(caminho_log, intervalo_s):
    """Extrai limite inferior, histórico amostrado e última melhoria diretamente do log do CBC."""
    if not caminho_log or not os.path.exists(caminho_log):
        return None, [], {"tempo_s": None, "custo": None}

    with open(caminho_log, "r", encoding="utf-8", errors="ignore") as arq:
        texto = arq.read()

    # 1) Limite inferior final (contorna bug do Pyomo 6.11 com CBC devel)
    limite_inf = None
    padroes_bound = [
        r"Bound:\s*([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)",
        r"\(best possible\s+([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)\)",
        r"Lower bound:\s*([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)",
    ]
    for padrao in padroes_bound:
        matches = re.findall(padrao, texto)
        if matches:
            val = numero_finito(matches[-1])
            if val is not None:
                limite_inf = val
                break

    # 2) Série temporal da tabela Branch-and-Bound (CBC devel e CBC 2.10 clássico)
    regex_devel = re.compile(
        r"^\s*(★|\*)?\s*(\d+)\s+(\d+)\s+(\d+)\s+"
        r"([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?|--|inf|-)\s+"
        r"(?:([A-Za-z0-9_/-]+)\s+)?"
        r"([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)\s+"
        r"([+-]?\d+(?:\.\d+)?%|--|inf)\s+"
        r"([+-]?\d+(?:\.\d+)?)\s*$"
    )
    regex_classico = re.compile(
        r"After\s+(\d+)\s+nodes,\s+\d+\s+on\s+tree,\s+"
        r"([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)\s+best\s+solution,\s+"
        r"best\s+possible\s+([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)\s+"
        r"\(([+-]?\d+(?:\.\d+)?)\s+seconds\)"
    )

    pontos = []
    melhor_info = {"tempo_s": None, "custo": None}
    melhor_custo_visto = None
    em_bb = False

    for linha in texto.splitlines():
        linha_low = linha.lower()
        if "branch and bound" in linha_low:
            em_bb = True
            continue
        if em_bb and ("postprocessing" in linha_low or linha.lstrip().startswith("✔")):
            em_bb = False

        m_dev = regex_devel.match(linha) if em_bb else None
        if m_dev:
            estrela, nos_str, _ontree, _depth, sol_str, _metodo, bnd_str, gap_str, tempo_str = m_dev.groups()
            gap_str = gap_str.rstrip("%")
            t_s = numero_finito(tempo_str)
            custo = numero_finito(sol_str)
            limite = numero_finito(bnd_str)
            gap_pct = numero_finito(gap_str)
            if gap_pct is None and custo is not None and limite is not None and abs(custo) > 1e-12:
                gap_pct = abs(custo - limite) / abs(custo) * 100.0
            if t_s is not None:
                pontos.append((t_s, custo, limite, gap_pct, int(nos_str)))
                if custo is not None and (
                    estrela is not None
                    or melhor_custo_visto is None
                    or custo < melhor_custo_visto - 1e-7
                ):
                    melhor_custo_visto = custo
                    melhor_info = {"tempo_s": t_s, "custo": custo}
            continue

        m_cls = regex_classico.search(linha)
        if m_cls:
            nos_str, sol_str, bnd_str, tempo_str = m_cls.groups()
            t_s = numero_finito(tempo_str)
            custo = numero_finito(sol_str)
            limite = numero_finito(bnd_str)
            gap_pct = (
                abs(custo - limite) / abs(custo) * 100.0
                if (custo is not None and limite is not None and abs(custo) > 1e-12)
                else None
            )
            if t_s is not None:
                pontos.append((t_s, custo, limite, gap_pct, int(nos_str)))
                if custo is not None and (melhor_custo_visto is None or custo < melhor_custo_visto - 1e-7):
                    melhor_custo_visto = custo
                    melhor_info = {"tempo_s": t_s, "custo": custo}

    # Reamostra os pontos do CBC respeitando INTERVALO_ACOMPANHAMENTO_MIN
    historico_cbc = []
    proxima_amostra_s = 0.0
    for t_s, custo, limite, gap_pct, nos_pt in pontos:
        if t_s >= proxima_amostra_s:
            historico_cbc.append({
                "Tempo (min)": t_s / 60.0,
                "Melhor custo viável (R$)": custo,
                "Limite inferior (R$)": limite,
                "Gap (%)": gap_pct,
                "Nós": nos_pt,
            })
            proxima_amostra_s = (int(t_s // intervalo_s) + 1) * intervalo_s

    return limite_inf, historico_cbc, melhor_info

if ACOMPANHAR_PROGRESSO and INTERVALO_ACOMPANHAMENTO_MIN <= 0:
    raise ValueError("INTERVALO_ACOMPANHAMENTO_MIN deve ser maior que zero.")

intervalo_amostragem_s = float(INTERVALO_ACOMPANHAMENTO_MIN) * 60.0

if acompanhar_com_highs:
    solver.options["mip_min_logging_interval"] = min(5.0, max(1.0, intervalo_amostragem_s))
    try:
        solver.set_instance(model)
        highs_model = getattr(solver, "_solver_model", None)
        if highs_model is None or not hasattr(highs_model, "cbMipLogging"):
            raise AttributeError("cbMipLogging indisponível nesta versão do highspy/Pyomo.")
    except Exception as erro_cb:
        print(
            f"[AVISO] Callback de progresso do HiGHS indisponível ({erro_cb}). "
            "Executando resolução padrão com tee=True."
        )
        acompanhar_com_highs = False
        highs_model = None

solve_kwargs = {"tee": True}
if warm_start_pronto:
    solve_kwargs["warmstart"] = True

caminho_log_cbc = None
if SOLVER == "cbc":
    fd_log, caminho_log_cbc = tempfile.mkstemp(prefix="cbc_eta_", suffix=".log")
    os.close(fd_log)
    solve_kwargs["logfile"] = caminho_log_cbc

resultado = None
erro_resolucao = None
limite_inferior_log_cbc = None

try:
    if acompanhar_com_highs:
        estado_amostragem = {"proxima_amostra_s": 0.0}

        def registrar_progresso(evento, *args):
            dados = evento.data_out if hasattr(evento, "data_out") else args[1]
            tempo_amostra_s = numero_finito(dados.running_time)
            if tempo_amostra_s is None or tempo_amostra_s < estado_amostragem["proxima_amostra_s"]:
                return

            custo = numero_finito(dados.mip_primal_bound)
            limite = numero_finito(dados.mip_dual_bound)
            gap = numero_finito(dados.mip_gap)
            nos_amostra = numero_finito(dados.mip_node_count)
            historico_progresso.append({
                "Tempo (min)": tempo_amostra_s / 60.0,
                "Melhor custo viável (R$)": custo,
                "Limite inferior (R$)": limite,
                "Gap (%)": gap * 100.0 if gap is not None else None,
                "Nós": int(nos_amostra) if nos_amostra is not None and nos_amostra >= 0 else None,
            })
            estado_amostragem["proxima_amostra_s"] = (
                int(tempo_amostra_s // intervalo_amostragem_s) + 1
            ) * intervalo_amostragem_s

        tem_cb_melhoria = hasattr(highs_model, "cbMipImprovingSolution")

        def registrar_melhoria(evento, *args):
            dados = evento.data_out if hasattr(evento, "data_out") else args[1]
            t_s = numero_finito(dados.running_time)
            custo = numero_finito(getattr(dados, "objective_function_value", None))
            if custo is None:
                custo = numero_finito(getattr(dados, "mip_primal_bound", None))
            if t_s is not None and custo is not None:
                melhor_incumbente_info["tempo_s"] = t_s
                melhor_incumbente_info["custo"] = custo

        highs_model.cbMipLogging += registrar_progresso
        if tem_cb_melhoria:
            highs_model.cbMipImprovingSolution += registrar_melhoria
        try:
            resultado = solver.solve(model, **solve_kwargs)
        except RuntimeError as erro_solve:
            erro_resolucao = erro_solve
        finally:
            highs_model.cbMipLogging -= registrar_progresso
            if tem_cb_melhoria:
                highs_model.cbMipImprovingSolution -= registrar_melhoria
    else:
        try:
            resultado = solver.solve(model, **solve_kwargs)
        except RuntimeError as erro_solve:
            erro_resolucao = erro_solve

    if SOLVER == "cbc":
        limite_inferior_log_cbc, hist_cbc, melhor_cbc = extrair_progresso_log_cbc(
            caminho_log_cbc, intervalo_amostragem_s
        )
        if ACOMPANHAR_PROGRESSO:
            historico_progresso.extend(hist_cbc)
        if melhor_cbc["tempo_s"] is not None:
            melhor_incumbente_info = melhor_cbc
finally:
    if caminho_log_cbc and os.path.exists(caminho_log_cbc):
        try:
            os.remove(caminho_log_cbc)
        except OSError:
            pass

tempo_execucao = time.perf_counter() - tempo_inicio

if resultado is not None:
    status_terminal = resultado.solver.termination_condition
    limite_inferior = numero_finito(resultado.problem.lower_bound)
    incumbente = numero_finito(resultado.problem.upper_bound)
else:
    status_terminal = f"Sem solução carregada ({erro_resolucao})"
    limite_inferior = None
    incumbente = None

if limite_inferior is None and limite_inferior_log_cbc is not None:
    limite_inferior = limite_inferior_log_cbc

obj_otimo = value(model.obj, exception=False)
solucao_disponivel = (
    resultado is not None
    and status_terminal in (
        TerminationCondition.optimal,
        TerminationCondition.feasible,
        TerminationCondition.maxTimeLimit,
    )
    and obj_otimo is not None
)

if incumbente is None and solucao_disponivel:
    incumbente = float(obj_otimo)

nos = None
if resultado is not None:
    try:
        nos_relatorio = resultado.solver.statistics.branch_and_bound.number_of_created_subproblems
        if nos_relatorio not in (None, float("inf")):
            nos = int(nos_relatorio)
    except Exception:
        pass

tempo_solver = None
if highs_model is not None:
    try:
        tempo_solver = numero_finito(highs_model.getRunTime())
        info = highs_model.getInfo()
        if info.mip_node_count >= 0:
            nos = int(info.mip_node_count)
        if limite_inferior is None:
            limite_inferior = numero_finito(info.mip_dual_bound)
        if incumbente is None and solucao_disponivel:
            incumbente = numero_finito(info.objective_function_value)
    except Exception:
        pass

if tempo_solver is None and resultado is not None:
    tempo_solver = numero_finito(getattr(resultado.solver, "wallclock_time", None))

tempo = tempo_solver if tempo_solver is not None else tempo_execucao

if limite_inferior is not None and incumbente is not None and abs(incumbente) > 1e-12:
    gap_final = abs(incumbente - limite_inferior) / abs(incumbente)
else:
    gap_final = None

print(f"\n[RESULTADOS DO {SOLVER.upper()} VIA PYOMO — {DESCRICAO_MODO}]")
print("Status:", status_terminal)
print(f"Função objetivo incumbente: R$ {incumbente:.4f}" if incumbente is not None else "Incumbente: não disponível")
print(f"Melhor limite inferior: R$ {limite_inferior:.4f}" if limite_inferior is not None else "Limite inferior: não disponível")
print(f"Gap relativo final: {gap_final:.2%}" if gap_final is not None else "Gap relativo final: não disponível")
print(f"Tempo (solver): {tempo:.2f} s ({tempo / 60.0:.2f} min)")
if tempo_solver is not None and abs(tempo_execucao - tempo_solver) > 1.0:
    print(f"Tempo total da célula (incl. interface Pyomo): {tempo_execucao:.2f} s")
if melhor_incumbente_info["tempo_s"] is not None:
    t_melhor = melhor_incumbente_info["tempo_s"]
    print(
        f"Última melhoria da incumbente: R$ {melhor_incumbente_info['custo']:.4f} "
        f"aos {t_melhor:.2f} s ({t_melhor / 60.0:.2f} min)"
    )
print(f"Nós explorados: {nos}" if nos is not None else "Nós explorados: não disponível")

if ACOMPANHAR_PROGRESSO:
    tempo_final_min = tempo / 60.0
    if historico_progresso:
        tempo_final_min = max(tempo_final_min, historico_progresso[-1]["Tempo (min)"])

    linha_final = {
        "Tempo (min)": tempo_final_min,
        "Melhor custo viável (R$)": float(incumbente) if incumbente is not None else None,
        "Limite inferior (R$)": float(limite_inferior) if limite_inferior is not None else None,
        "Gap (%)": gap_final * 100.0 if gap_final is not None else None,
        "Nós": int(nos) if nos is not None else None,
    }
    if not historico_progresso or (
        linha_final["Tempo (min)"] - historico_progresso[-1]["Tempo (min)"] > 1e-2
    ):
        historico_progresso.append(linha_final)
    else:
        historico_progresso[-1] = linha_final

    tabela_progresso = pd.DataFrame(historico_progresso)
    display(tabela_progresso.round({
        "Tempo (min)": 2,
        "Melhor custo viável (R$)": 2,
        "Limite inferior (R$)": 2,
        "Gap (%)": 3,
    }))

# Resumo de custos e comparação segura com o cenário-base (não quebra se a célula 4 não foi executada)
if solucao_disponivel:
    print(f"\n[SUCESSO] Solução encontrada pelo {SOLVER.upper()} via Pyomo!")
    consumo_otimizado_kwh = sum(
        param_bombas[p][0] * FATOR_HORA * value(model.x[p, t])
        for p in model.P for t in model.T
    )
    custo_otimizado_real = sum(
        C[(t * DURACAO_PASSO_MIN) // 60] * param_bombas[p][0] * FATOR_HORA * value(model.x[p, t])
        for p in model.P for t in model.T
    )
    partidas_otimizado = contar_partidas(model)

    base_ok = base_esta_disponivel()
    linhas_comparacao = []
    if base_ok:
        linhas_comparacao.append({
            "Cenário": f"Base (tarifa fixa · {MODO_TEMPORAL})",
            "Energia (kWh)": round(consumo_base_kwh, 2),
            "Custo com tarifa real (R$)": round(custo_base_real, 2),
            "Partidas das bombas": globals().get("partidas_base"),
        })
    linhas_comparacao.append({
        "Cenário": f"Otimização (tarifa horária · {MODO_TEMPORAL})",
        "Energia (kWh)": round(consumo_otimizado_kwh, 2),
        "Custo com tarifa real (R$)": round(custo_otimizado_real, 2),
        "Partidas das bombas": partidas_otimizado,
    })
    comparacao_custos = pd.DataFrame(linhas_comparacao)
    display(comparacao_custos)

    if base_ok:
        economia = custo_base_real - custo_otimizado_real
        percentual_economia = 100.0 * economia / custo_base_real if custo_base_real else 0.0
        print(f"Economia financeira: R$ {economia:.2f} ({percentual_economia:.2f}%)")
    else:
        print(
            "Cenário-base indisponível (desativado, não executado ou sem solução viável); "
            "a comparação de custos foi omitida."
        )
else:
    print(f"\nModelo inviável ou sem solução viável encontrada pelo {SOLVER.upper()}.")

## 6. Gráficos e Análise Visual

Gera os perfis de bombeamento, a dinâmica dos reservatórios (em m³ e %) para a otimização tarifária (e para o cenário-base, se disponível) e, com `ACOMPANHAR_PROGRESSO = True`, os gráficos separados da evolução de **Incumbente vs. Limite Inferior** e **Gap (%)** ao longo do tempo.

In [ ]:
# Gráficos adaptados automaticamente ao MODO_TEMPORAL ("hora" ou "minuto")
# + gráficos separados de progressão (incumbente/limite inferior e gap) quando ACOMPANHAR_PROGRESSO=True.
if globals().get("solucao_disponivel", False):
    tempo_horas_com_fim = [t * DURACAO_PASSO_MIN / 60.0 for t in range(N_PASSOS + 1)]
    preco_energia = [C[(t * DURACAO_PASSO_MIN) // 60] for t in passos] + [C[23]]
    cores_res = ["#2E7D32", "#689F38", "#AFB42B", "#FBC02D", "#F57C00"]
    marcador_res = "o" if MODO_TEMPORAL == "hora" else None
    tamanho_marcador = 4 if MODO_TEMPORAL == "hora" else None

    def plotar_graficos_cenario(modelo_cenario, nome_cenario):
        # q está sempre em m³/min; multiplicar por 60 converte para m³/h em ambos os modos.
        vazao_total = [
            60.0 * sum(value(modelo_cenario.q[p, t]) for p in bombas)
            for t in passos
        ]
        vazao_degrau = vazao_total + [vazao_total[-1]]

        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

        ax1.fill_between(
            tempo_horas_com_fim, 0, vazao_degrau, step="post",
            color="#2196F3", alpha=0.75, label="Vazão bombeada (m³/h)",
        )
        ax1.axhline(
            Q_ETA_max, color="#0b5394", linestyle="--", linewidth=1.2, alpha=0.8,
            label=f"Capacidade da ETA ({Q_ETA_max} m³/h)",
        )
        ax1.set_ylabel("Vazão total (m³/h)", color="#0b5394", fontweight="bold")
        ax1.tick_params(axis="y", labelcolor="#0b5394")
        ax1.set_ylim(0, Q_ETA_max * 1.12)
        ax1.set_title(
            f"Perfil de bombeamento e tarifa — {nome_cenario} ({DESCRICAO_MODO})",
            fontsize=13.5,
        )
        ax1.grid(True, linestyle="--", alpha=0.4)

        ax1_twin = ax1.twinx()
        ax1_twin.step(
            tempo_horas_com_fim, preco_energia, where="post",
            color="#d32f2f", linewidth=2, label="Tarifa de energia (R$/kWh)",
        )
        ax1_twin.set_ylabel("Tarifa (R$/kWh)", color="#990000", fontweight="bold")
        ax1_twin.tick_params(axis="y", labelcolor="#990000")
        ax1.axvspan(18, 21, color="red", alpha=0.1, label="Horário de ponta (18h–21h)")

        h1, l1 = ax1.get_legend_handles_labels()
        h2, l2 = ax1_twin.get_legend_handles_labels()
        ax1.legend(h1 + h2, l1 + l2, loc="upper left", bbox_to_anchor=(1.08, 1.0))

        for r in reservatorios:
            volume = [param_res[r][2]] + [
                value(modelo_cenario.V[r, t]) for t in passos
            ]
            ax2.plot(
                tempo_horas_com_fim, volume, linewidth=1.6,
                marker=marcador_res, markersize=tamanho_marcador,
                color=cores_res[r - 1], label=f"Reservatório {r}",
            )
        ax2.axvspan(18, 21, color="red", alpha=0.1)
        ax2.set_xlabel("Hora do dia (0 = início, 24 = final)", fontweight="bold")
        ax2.set_ylabel("Volume armazenado (m³)", fontweight="bold")
        ax2.set_title(
            f"Dinâmica de carga e descarga dos reservatórios ({DESCRICAO_MODO})",
            fontsize=13.5,
        )
        ax2.set_xticks(range(0, 25, 2))
        ax2.set_xlim(0, 24)
        ax2.legend(loc="upper left", bbox_to_anchor=(1.08, 1.0))
        ax2.grid(True, linestyle="--", alpha=0.5)
        fig.tight_layout()
        plt.show()

        fig_pct, ax_pct = plt.subplots(figsize=(12, 6))
        for r in reservatorios:
            V_max = param_res[r][1]
            volume = [param_res[r][2]] + [
                value(modelo_cenario.V[r, t]) for t in passos
            ]
            nivel_percentual = [100.0 * v / V_max for v in volume]
            ax_pct.plot(
                tempo_horas_com_fim, nivel_percentual, linewidth=1.6,
                marker=marcador_res, markersize=tamanho_marcador,
                color=cores_res[r - 1], label=f"Reservatório {r}",
            )
        ax_pct.axvspan(
            18, 21, color="red", alpha=0.1,
            label="Horário de ponta (18h00–20h59)",
        )
        ax_pct.axhline(100, color="black", linestyle=":", alpha=0.5, label="Capacidade máxima (100%)")
        ax_pct.axhline(15, color="#b71c1c", linestyle="--", alpha=0.5, label="Reserva mínima (15%)")
        ax_pct.set_title(
            f"Nível dos reservatórios (%) — {nome_cenario} ({DESCRICAO_MODO})",
            fontsize=13.5,
        )
        ax_pct.set_xlabel("Hora do dia (0 = início, 24 = final)", fontweight="bold")
        ax_pct.set_ylabel("Volume armazenado (%)", fontweight="bold")
        ax_pct.set_xticks(range(0, 25, 2))
        ax_pct.set_xlim(0, 24)
        ax_pct.set_ylim(0, 105)
        ax_pct.grid(True, linestyle="--", alpha=0.7)
        ax_pct.legend(bbox_to_anchor=(1.02, 1.0), loc="upper left")
        fig_pct.tight_layout()
        plt.show()

    def plotar_progresso_otimizacao(df_prog):
        """Plota gráficos separados da evolução de incumbente/limite inferior e do gap."""
        if df_prog is None or df_prog.empty:
            print("Nenhum dado de progresso disponível para plotar.")
            return

        df_plot = df_prog.copy()
        tempo_max_min = float(df_plot["Tempo (min)"].max())
        usar_segundos = tempo_max_min < 2.0
        eixo_x = df_plot["Tempo (min)"] * (60.0 if usar_segundos else 1.0)
        rotulo_x = "Tempo de execução do solver (s)" if usar_segundos else "Tempo de execução do solver (min)"

        # Filtra pontos iniciais com limite inferior == 0 (antes do LP raiz) para não achatar o eixo Y
        mask_primal = df_plot["Melhor custo viável (R$)"].notna()
        mask_dual = df_plot["Limite inferior (R$)"].notna() & (df_plot["Limite inferior (R$)"] > 0)
        mask_gap = df_plot["Gap (%)"].notna() & (df_plot["Limite inferior (R$)"] > 0)

        # Gráfico 1: Incumbente vs. Limite inferior ao longo do tempo
        fig_lim, ax_lim = plt.subplots(figsize=(12, 4.8))
        if mask_primal.any():
            ax_lim.plot(
                eixo_x[mask_primal], df_plot.loc[mask_primal, "Melhor custo viável (R$)"],
                marker="o", linewidth=2.0, color="#006699", label="Melhor custo viável — Incumbente (R$)",
            )
        if mask_dual.any():
            ax_lim.plot(
                eixo_x[mask_dual], df_plot.loc[mask_dual, "Limite inferior (R$)"],
                marker="s", linewidth=2.0, color="#15803D", label="Limite inferior — Dual Bound (R$)",
            )
        ax_lim.set_title(
            f"Progressão da otimização ({SOLVER.upper()} · {DESCRICAO_MODO}) — Incumbente vs. Limite Inferior",
            fontsize=13,
        )
        ax_lim.set_xlabel(rotulo_x, fontweight="bold")
        ax_lim.set_ylabel("Custo diário (R$)", fontweight="bold")
        ax_lim.grid(True, linestyle="--", alpha=0.5)
        ax_lim.legend(loc="best")
        fig_lim.tight_layout()
        plt.show()

        # Gráfico 2: Evolução do Gap (%) ao longo do tempo
        fig_gap, ax_gap = plt.subplots(figsize=(12, 4.5))
        if mask_gap.any():
            ax_gap.plot(
                eixo_x[mask_gap], df_plot.loc[mask_gap, "Gap (%)"],
                marker="o", linewidth=2.0, color="#D96B00", label="Gap relativo (%)",
            )
        gap_alvo_pct = obter_gap_alvo_pct()
        if gap_alvo_pct is not None:
            ax_gap.axhline(
                gap_alvo_pct, color="#15803D", linestyle="--", linewidth=1.4,
                label=f"Meta de gap-alvo ({gap_alvo_pct:.2f}%)",
            )
        ax_gap.set_title(
            f"Progressão da otimização ({SOLVER.upper()} · {DESCRICAO_MODO}) — Evolução do Gap (%)",
            fontsize=13,
        )
        ax_gap.set_xlabel(rotulo_x, fontweight="bold")
        ax_gap.set_ylabel("Gap relativo (%)", fontweight="bold")
        ax_gap.set_ylim(bottom=0)
        ax_gap.grid(True, linestyle="--", alpha=0.5)
        ax_gap.legend(loc="best")
        fig_gap.tight_layout()
        plt.show()

    plotar_graficos_cenario(model, "Otimização tarifária")
    if base_esta_disponivel():
        plotar_graficos_cenario(modelo_base, "Cenário-base (tarifa fixa)")
    if ACOMPANHAR_PROGRESSO:
        plotar_progresso_otimizacao(globals().get("tabela_progresso"))
else:
    print("Execute a otimização com sucesso antes de gerar os gráficos.")